# Extract & Label Chess Piece Glyphs from PDF

**Workflow:**
1. Load previous classifier (optional) to pre-filter candidates
2. Extract candidate glyphs from PDF
3. Manually label ambiguous ones
4. Train new classifier
5. Download zip with labeled glyphs + classifier

## Step 1 — Mount Google Drive and load PDF

In [ ]:
from google.colab import drive
import os

drive.mount('/content/gdrive')

In [ ]:
PDF_PATH = '/content/gdrive/MyDrive/chess_book.pdf'

if not os.path.exists(PDF_PATH):
    print(f'❌ PDF not found: {PDF_PATH}')
else:
    size_mb = os.path.getsize(PDF_PATH) / (1024*1024)
    print(f'✅ PDF loaded: {PDF_PATH}  ({size_mb:.1f} MB)')

## Step 2 — Install dependencies

In [ ]:
!apt-get install -y poppler-utils
!pip install -q pdfplumber pdf2image pillow scikit-learn scikit-image opencv-python

## Step 3 — Configuration

In [ ]:
import pdfplumber
from pdf2image import convert_from_path
from PIL import Image
import os
from pathlib import Path
import pickle
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from skimage import filters, transform
import zipfile

START_PAGE = 1
END_PAGE = 20
GLYPH_DPI = 150
IMG_SIZE = 32
CONFIDENCE_THRESHOLD = 0.7

GLYPHS_DIR = './glyphs_results/glyphs'
WORK_DIR = './glyphs_results'
PIECE_CLASSES = ['K', 'Q', 'R', 'B', 'N']

Path(WORK_DIR).mkdir(exist_ok=True)
Path(GLYPHS_DIR).mkdir(exist_ok=True)
for piece in PIECE_CLASSES:
    Path(f'{GLYPHS_DIR}/{piece}').mkdir(exist_ok=True)

print(f'✅ Output directory: {WORK_DIR}/')
print(f'   Structure: glyphs_results/glyphs/{{K,Q,R,B,N}}')

## Step 3.5 — Load previous classifier (optional)

In [ ]:
from google.colab import files

classifier = None
previous_labeled_count = {piece: 0 for piece in PIECE_CLASSES}

print('📁 Upload a previous classifier zip (optional, press Skip if none):')
print('   Classifier will be used to pre-filter candidates')
print('   (Old labeled glyphs will not be carried forward)')
print()

try:
    uploaded = files.upload()
    print(f'Uploaded files: {list(uploaded.keys())}')
    
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            print(f'Extracting {filename}...')
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('./previous')
            
            classifier_path = './previous/glyphs_results/classifier.pkl'
            if os.path.exists(classifier_path):
                print(f'✅ Found classifier.pkl, loading...')
                with open(classifier_path, 'rb') as f:
                    classifier = pickle.load(f)
                print(f'✅ Classifier loaded successfully')
            else:
                print(f'❌ classifier.pkl not found')
            
            for piece in PIECE_CLASSES:
                path = f'./previous/glyphs_results/glyphs/{piece}'
                if os.path.exists(path):
                    count = len([f for f in os.listdir(path) if f.endswith('.png')])
                    previous_labeled_count[piece] = count
            
            total_prev = sum(previous_labeled_count.values())
            if total_prev > 0:
                print(f'✅ Previous session had {total_prev} labeled glyphs (info only)')
except Exception as e:
    print(f'❌ Error loading zip: {e}')
    import traceback
    traceback.print_exc()

## Step 4 — Extract piece templates from sprite image

In [ ]:
import cv2

def extract_image_features(img):
    if img.width < 5 or img.height < 5:
        return None
    img_resized = transform.resize(np.array(img), (IMG_SIZE, IMG_SIZE), anti_aliasing=True)
    features = []
    features.append(np.mean(img_resized))
    features.append(np.std(img_resized))
    features.append(img.width / max(img.height, 1))
    edges = filters.sobel(img_resized)
    features.append(np.mean(edges))
    features.append(np.mean(np.sum(img_resized, axis=0)))
    features.append(np.mean(np.sum(img_resized, axis=1)))
    return np.array(features)

def render_page(pdf_path, page_num, dpi=150):
    images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1, dpi=dpi)
    return images[0] if images else None

print('📁 Loading chess pieces sprite image...')
sprite_path = '/content/gdrive/MyDrive/entrainement_ocr_echecs/Chess_Pieces_Sprite.png'

sprite_rgba = cv2.imread(sprite_path, cv2.IMREAD_UNCHANGED)
sprite_gray = cv2.imread(sprite_path, cv2.IMREAD_GRAYSCALE)

if sprite_gray is None:
    print(f'❌ Could not load sprite from {sprite_path}')
    templates = {}
    templates_display = {}
    piece_order = []
else:
    print(f'✅ Sprite loaded: {sprite_gray.shape}')

    # Sprite has 2 rows: top = white pieces, bottom = black pieces
    # Left-to-right order per row: K, Q, B, N, R, P
    piece_order = ['K', 'Q', 'B', 'N', 'R', 'P']
    h, w = sprite_gray.shape
    num_cols = len(piece_order)
    piece_width = w // num_cols
    row_height = h // 2  # top half = white pieces

    templates = {}
    templates_display = {}

    for idx, piece_name in enumerate(piece_order):
        x_start = idx * piece_width
        x_end = x_start + piece_width
        templates[piece_name] = sprite_gray[0:row_height, x_start:x_end]
        if sprite_rgba is not None and sprite_rgba.ndim == 3 and sprite_rgba.shape[2] == 4:
            crop_bgra = sprite_rgba[0:row_height, x_start:x_end]
            crop_rgba = cv2.cvtColor(crop_bgra, cv2.COLOR_BGRA2RGBA)
            templates_display[piece_name] = Image.fromarray(crop_rgba, 'RGBA')
        else:
            templates_display[piece_name] = Image.fromarray(templates[piece_name]).convert('RGBA')

    print(f'✅ Extracted {len(templates)} white piece templates ({piece_width}x{row_height}px each)')

In [ ]:
print('=' * 70)
print('EXTRACTED PIECE TEMPLATES')
print('=' * 70)
print(f'\nTotal templates extracted: {len(templates)}')
print('\nDisplaying each piece template:\n')

for piece_name in piece_order:
    if piece_name not in templates_display:
        continue

    template = templates[piece_name]
    h, w = template.shape

    print(f'\n📍 Piece: {piece_name}  Size: {w}x{h}')

    pil_template = templates_display[piece_name]
    pil_template_large = pil_template.resize((w*3, h*3), Image.NEAREST)

    display(pil_template_large)

print('\n' + '=' * 70)
print('✅ Do these pieces look correct?')
print('   - YES: proceed to Step 5 (template matching on PDF)')
print('   - NO: check sprite image arrangement or adjust piece_order')
print('=' * 70)

In [ ]:
glyph_words = []

if not templates:
    print('❌ No templates available — run Step 4 first')
else:
    print(f'🔍 Matching {len(templates)} piece templates against PDF pages...')
    with pdfplumber.open(PDF_PATH) as pdf:
        pdf_page_count = len(pdf.pages)
        end_page = min(END_PAGE, pdf_page_count)
        
        for page_idx in range(START_PAGE - 1, end_page):
            page_num = page_idx + 1
            page_image = render_page(PDF_PATH, page_idx, dpi=GLYPH_DPI)
            if page_image is None:
                continue
            
            page_cv = cv2.cvtColor(np.array(page_image), cv2.COLOR_RGB2GRAY)
            
            for piece_name, template in templates.items():
                if template.shape[0] == 0 or template.shape[1] == 0:
                    continue
                
                result = cv2.matchTemplate(page_cv, template, cv2.TM_CCOEFF)
                threshold = np.percentile(result, 90)
                
                locs = np.where(result >= threshold)
                th, tw = template.shape
                for y, x in zip(locs[0], locs[1]):
                    crop = page_image.crop((x, y, x+tw, y+th))
                    features = extract_image_features(crop)
                    if features is None:
                        continue
                    
                    is_duplicate = any(
                        abs(x - e['bbox'][0]) < tw/2 and abs(y - e['bbox'][1]) < th/2
                        for e in glyph_words
                    )
                    if not is_duplicate:
                        glyph_words.append({
                            'page': page_num,
                            'text': piece_name,
                            'bbox': (x, y, x+tw, y+th),
                            'crop': crop,
                            'features': features,
                            'source': 'template',
                        })
    
    print(f'✅ Found {len(glyph_words)} chess piece candidates')
    print(f'Classifier available: {classifier is not None}')
    
    if classifier is not None:
        filtered_words = []
        for word in glyph_words:
            conf = np.max(classifier.predict_proba([word['features']])[0])
            word['confidence'] = conf
            if conf >= CONFIDENCE_THRESHOLD:
                filtered_words.append(word)
        print(f'⚡ Classifier pre-filtered to {len(filtered_words)} candidates')
        glyph_words = filtered_words
    else:
        print('ℹ️  No classifier available; showing all candidates')

## Step 4.8 — Match templates against PDF pages

## Step 4.5 — Verify extracted piece templates

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

current_idx = 0
saved_count = {piece: 0 for piece in PIECE_CLASSES}
skipped = 0
discarded = 0

def show_glyph(idx):
    global current_idx, saved_count, skipped, discarded
    if idx >= len(glyph_words):
        clear_output()
        print(f'✅ Labeling complete!')
        print(f'Saved:')
        for piece in PIECE_CLASSES:
            print(f'  {piece}: {saved_count[piece]}')
        print(f'  Skipped: {skipped}')
        print(f'  Discarded: {discarded}')
        return
    current_idx = idx
    word_info = glyph_words[idx]
    crop = word_info['crop']
    clear_output()
    conf_text = f" (confidence: {word_info.get('confidence', 0):.2%})" if 'confidence' in word_info else ""
    print(f'Glyph {idx + 1}/{len(glyph_words)} — P{word_info["page"]}: "{word_info["text"]}"{conf_text}')
    print()
    display(crop)
    print()
    def save_glyph(piece):
        count = saved_count[piece]
        filename = f'{GLYPHS_DIR}/{piece}/{count + 1:04d}.png'
        crop.save(filename)
        saved_count[piece] += 1
        show_glyph(idx + 1)
    def skip():
        global skipped
        skipped += 1
        show_glyph(idx + 1)
    def discard():
        global discarded
        discarded += 1
        show_glyph(idx + 1)
    buttons = [
        widgets.Button(description='K (King)', button_style='info'),
        widgets.Button(description='Q (Queen)', button_style='info'),
        widgets.Button(description='R (Rook)', button_style='info'),
        widgets.Button(description='B (Bishop)', button_style='info'),
        widgets.Button(description='N (Knight)', button_style='info'),
        widgets.Button(description='Skip', button_style='warning'),
        widgets.Button(description='❌ Discard', button_style='danger'),
    ]
    buttons[0].on_click(lambda _: save_glyph('K'))
    buttons[1].on_click(lambda _: save_glyph('Q'))
    buttons[2].on_click(lambda _: save_glyph('R'))
    buttons[3].on_click(lambda _: save_glyph('B'))
    buttons[4].on_click(lambda _: save_glyph('N'))
    buttons[5].on_click(lambda _: skip())
    buttons[6].on_click(lambda _: discard())
    display(widgets.HBox(buttons))

if len(glyph_words) > 0:
    show_glyph(0)
else:
    print('❌ No candidates to label')

## Step 6 — Train classifier

In [ ]:
X_train = []
y_train = []
for piece_idx, piece in enumerate(PIECE_CLASSES):
    path = f'{GLYPHS_DIR}/{piece}'
    if os.path.exists(path):
        for img_file in os.listdir(path):
            if img_file.endswith('.png'):
                img = Image.open(f'{path}/{img_file}').convert('L')
                features = extract_image_features(img)
                if features is not None:
                    X_train.append(features)
                    y_train.append(piece_idx)

if len(X_train) > 10:
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    
    num_samples = len(X_train)
    
    n_estimators = max(50, min(200, 50 + (num_samples // 5)))
    max_depth = min(15 + (num_samples // 20), 30)
    min_samples_split = max(2, num_samples // 10)
    
    new_classifier = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    new_classifier.fit(X_train, y_train)
    
    with open(f'{WORK_DIR}/classifier.pkl', 'wb') as f:
        pickle.dump(new_classifier, f)
    
    print(f'✅ Trained classifier on {num_samples} labeled glyphs')
    print(f'   Accuracy: {new_classifier.score(X_train, y_train):.1%}')
    print(f'   Parameters:')
    print(f'     n_estimators: {n_estimators}')
    print(f'     max_depth: {max_depth}')
    print(f'     min_samples_split: {min_samples_split}')
else:
    print('⚠️  Not enough labeled samples (need ≥10)')

## Step 7 — Export zip with classifier

In [ ]:
import shutil

zip_filename = 'chess_glyphs_classifier.zip'
shutil.make_archive('chess_glyphs_classifier', 'zip', '.', WORK_DIR)

final_counts = {}
total_glyphs = 0
for piece in PIECE_CLASSES:
    path = f'{GLYPHS_DIR}/{piece}'
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.endswith('.png')])
        final_counts[piece] = count
        total_glyphs += count

print(f'✅ Export complete!')
print(f'Zip file: {zip_filename}')
print(f'Structure:')
print(f'  glyphs_results/')
print(f'  ├── classifier.pkl')
print(f'  └── glyphs/')
for piece in PIECE_CLASSES:
    count = final_counts.get(piece, 0)
    print(f'      ├── {piece}/ ({count} images)')
print(f'Total: {total_glyphs} labeled glyphs')

from google.colab import files
files.download(zip_filename)